# Point Cloud to CAD-sequence

In this notebook the complete interactive pipeline for encoding point clouds into a latent space, from which DeepCAD decodes a CAD-sequence.

In [1]:
import os
import sys
import importlib

import torch

sys.path.append("..")
sys.path.append("../code")
from dataset import PointCloudEmbeddingSequenceDataset
from models.DeepCAD.config.configAE import ConfigAE
from models.DeepCAD.trainer.trainerAE import TrainerAE

### Variables

Store the models in ```experiments```, a results directory will be created for each respective model.

In [2]:
model_name = "best"

### Constants

In [3]:
model_path = os.path.join("experiments", model_name) + ".pth"
results_dir = os.path.join("experiments", model_name + "_results")
if not os.path.exists(results_dir):
    os.mkdir(results_dir)
latent_dim = 256

### Create and load pre-trained PointNet++

In [4]:
def inplace_relu(m):
    classname = m.__class__.__name__
    if classname.find('ReLU') != -1:
        m.inplace=True

sys.path.append(os.path.join('..', 'models','Pointnet_Pointnet2_pytorch', 'models'))
model = importlib.import_module('pointnet2_cls_ssg')
classifier = model.get_model(latent_dim, normal_channel=False)
criterion = model.get_loss_mse()
classifier.apply(inplace_relu)

saved_model = torch.load(model_path, map_location=torch.device('cpu'), weights_only=True)
state_dict = saved_model['model_state_dict']
if 'module.' in next(iter(state_dict)):
    monitor.log_and_print("Model was saved wrapped in nn.DataParallel.\nRemoving 'module.' from state dict.")
    state_dict = {k.replace('module.', ''): v for k, v in state_dict.items()}#
classifier.eval()
classifier.load_state_dict(state_dict)

<All keys matched successfully>

### Create and load pre-trained DeepCAD

In [5]:
cfg = ConfigAE('test', model_path="../data/latent")
tr_agent = TrainerAE(cfg)
tr_agent.net.eval()
tr_agent.load_ckpt(cfg.ckpt)

Loading checkpoint from /Users/saidharb/Documents/LocalDocuments/Master-Thesis/Point-Cloud-Reconstruction/data/latent/pretrained/model/ckpt_epoch1000.pth ...


### Load data

In [61]:
def get_data(indices, dataset):
    pc_list = []        
    lat_rep_list = []  
    cad_seq_list = []   
    for i in indices:
        pc, lat_rep, cad_seq = dataset[i]
        pc_list.append(pc)
        lat_rep_list.append(lat_rep)
        cad_seq_list.append(cad_seq)
    pc_batch = torch.stack(pc_list, dim=0)
    lat_rep_batch = torch.stack(lat_rep_list, dim=0)
    cad_seq_batch = torch.stack(cad_seq_list, dim=0)
    return pc_batch, lat_rep_batch, cad_seq_batch

In [62]:
def infer_pointnet(indices, dataset):
    pc, lat_rep, cad_seq = get_data(indices, dataset)
    with torch.no_grad():
        pc = pc.transpose(2, 1)
        pred, _ = classifier(pc)
        loss = criterion(pred, lat_rep)
        print(f"Avg. MSE-Loss: {loss.detach().item():.5f}")
        return pred, cad_seq

In [63]:
def infer_deepcad(pred, cad_seq):
    with torch.no_grad():
        pred = pred.unsqueeze(dim = 1)
        output = tr_agent.decode(pred)
        output["tgt_commands"] = cad_seq[:, :, 0] 
        output["tgt_args"] = cad_seq[:, :, 1:]
        loss_dict = tr_agent.loss_func(output)
        batch_out_vec = tr_agent.logits2vec(output)
        print(f"Avg. Command-Loss: {loss_dict['loss_cmd'].detach().cpu().item():.5f}")
        print(f"Avg. Argument-Loss: {loss_dict['loss_args'].detach().cpu().item():.5f}")

In [68]:
dataset = PointCloudEmbeddingSequenceDataset("../data", 'test')
print(f"Dataset contains {len(dataset)} samples.")

Dataset contains 8038 samples.


In [72]:
indices = range(1,12)

In [73]:
pred, trgt_cad_seq = infer_pointnet(indices, dataset)

Avg. MSE-Loss: 0.09193


In [74]:
infer_deepcad(pred, trgt_cad_seq)

Avg. Command-Loss: 12.90870
Avg. Argument-Loss: 17.38873


### Gedanken

- ich sollte hier ein Ordner haben in den ich das zu benutzende model setze
- dort werden auch die ergebnisse/visualisierungen abgespeichert
- visulization of CAD commands
- export2step also here
- also retrieve the pc to be able to quickly show it in cloud compare